In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler


In [20]:
df = pd.read_csv('../realises/data.csv')

In [21]:
df = df.drop(columns=['Unnamed: 0'])
df.head(3)

,id,rooms,metro,area,floor,parking,price,renovation,room_area,balcony,windows,bathroom,allowed_children_pets,ceiling_height,city
0,273614615,2,Арбатская,58.0,12,unknown,225000.0,Евроремонт,19.000000,unknown,На улицу и двор,Совмещенный (2),unknown,3.9,Москва
1,274475342,3,Смоленская,98.0,2,подземная,250000.0,Евроремонт,21.000000,unknown,Во двор,"Совмещенный (1), Раздельный (1)","Можно с детьми, Можно с животными",3.2,Москва
2,273973191,3,Смоленская,120.0,5,открытая,130000.0,Евроремонт,31.666667,unknown,На улицу,Совмещенный (1),Можно с животными,3.0,Москва


In [22]:
# делим на численные и категориальные
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns
# кодируем категориальные
encoder = OneHotEncoder()
encoded = encoder.fit_transform(df[categorical_cols])
encoded_df = pd.DataFrame(encoded.toarray(), columns=encoder.get_feature_names_out(categorical_cols))
# кодируем численные
scaler = StandardScaler()
scaled = scaler.fit_transform(df[numeric_cols])
scaled_df = pd.DataFrame(scaled, columns=numeric_cols)
# объединяем в один фрейм
encoded_df_final = pd.concat([encoded_df, scaled_df], axis=1)
encoded_df_final.head(4)
# encoded_df_final.to_csv('Analytical-mishanina_data.csv', encoding='utf-8')

,metro_unknown,metro_Авиамоторная,metro_Автово,metro_Автозаводская,metro_Адмиралтейская,metro_Академическая,metro_Александровский сад,metro_Алексеевская,metro_Алма-Атинская,metro_Алтуфьево,...,city_Краснодарский край,city_Москва,city_Санкт-Петербург,id,rooms,area,floor,price,room_area,ceiling_height
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.318409,0.090646,-0.004972,0.824141,3.052093,-0.006775,0.166618
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.363722,1.203787,1.181425,-0.952226,3.534567,-0.006775,0.038428
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.337286,1.203787,1.833944,-0.419316,1.218691,-0.006775,0.001802
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.280809,2.316928,0.944146,-0.952226,2.762608,-0.006775,56.222207


In [23]:
# другой способ кодировки
encoded_df_final_2 =  pd.get_dummies(df)
encoded_df_final_2.head(5)
# encoded_df_final_2.to_csv('Analytical-mishanina_data2.csv', encoding='utf-8')

,id,rooms,area,floor,price,room_area,ceiling_height,metro_unknown,metro_Авиамоторная,metro_Автово,...,"bathroom_Совмещенный (3), Раздельный (1)","bathroom_Совмещенный (3), Раздельный (3)",bathroom_Совмещенный (4),allowed_children_pets_unknown,allowed_children_pets_Можно с детьми,"allowed_children_pets_Можно с детьми, Можно с животными",allowed_children_pets_Можно с животными,city_Краснодарский край,city_Москва,city_Санкт-Петербург
0,273614615,2,58.0,12,225000.0,19.000000,3.9,False,False,False,...,False,False,False,True,False,False,False,False,True,False
1,274475342,3,98.0,2,250000.0,21.000000,3.2,False,False,False,...,False,False,False,False,False,True,False,False,True,False
2,273973191,3,120.0,5,130000.0,31.666667,3.0,False,False,False,...,False,False,False,False,False,False,True,False,True,False
3,272900409,4,90.0,2,210000.0,16.500000,310.0,False,False,False,...,False,False,False,False,True,False,False,False,True,False
4,271036964,4,170.0,6,290000.0,23.500000,3.2,False,False,False,...,False,False,False,False,True,False,False,False,True,False


In [24]:
# 3 вариант в numeric_cols удаляем колонку price, по которой обучаем модель
# делим на численные и категориальные
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.drop('price') 
categorical_cols = df.select_dtypes(include=['object']).columns
# кодируем категориальные
encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(df[categorical_cols])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(categorical_cols))
# кодируем численные
scaler = StandardScaler()
scaled = scaler.fit_transform(df[numeric_cols])
scaled_df = pd.DataFrame(scaled, columns=numeric_cols)
# объединяем в один датафрейм
encoded_df_final = pd.concat([encoded_df, scaled_df], axis=1)

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error
import pandas as pd
# Целевая переменная
y = df['price']
# Делим выборку
X_train, X_test, y_train, y_test = train_test_split(encoded_df_final, y, test_size=0.2, random_state=42)
# Обучаем модель
model = LinearRegression()
model.fit(X_train, y_train)
# Предсказания
y_pred = model.predict(X_test)
# Считаем MAPE
mape = mean_absolute_percentage_error(y_test, y_pred) * 100
print(f"MAPE: {mape:.2f}%")

MAPE: 21.07%


In [26]:
# # 4 вариант с get_dummies и в numeric_cols удаляем колонку price, по которой обучаем модель
# encoded_df_final_2 = pd.get_dummies(df)
# # кодируем все категориальные переменные с помощью get_dummies
# encoded_df_final_2 = pd.get_dummies(df.drop(columns=['id']))  # убираем id, он не нужен

In [27]:
# # целевая переменная
# y = df['price']
# # признаки (после кодировки)
# X = encoded_df_final_2.drop(columns=['price'])
# # делим на тренировочную и тестовую выборки
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# # обучаем модель
# model = LinearRegression()
# model.fit(X_train, y_train)
# # делаем предсказания
# y_pred = model.predict(X_test)
# # считаем MAPE
# mape = mean_absolute_percentage_error(y_test, y_pred) * 100
# print(f"MAPE: {mape:.2f}%")